# 🧬 NB02 — DAE + Discrete Hazard (v2.2-ram · Multi-Cohort)
**Stage:** NB02 | **RAM strategy:** sequential cohorts, float32, del+gc per fold

### What changed from v2.1-ram → v2.2-ram
| Change | Where |
|--------|-------|
| `DiscreteLifespanLoss` fully vectorised — O(N·K) Python loop → 3 tensor ops | Cell 4 |
| Vectorised loss is `torch.autocast`-compatible (fp16 safe) | Cell 4 |
| Risk score at inference changed from `-hazard.mean()` → `-expected_bin` (matches NB04) | Cell 5 |
| Version bump NB_VERSION 2.1-ram → 2.2-ram | Cell 1 |
| All v2.1 RAM guards kept (float32, nlargest, del+gc, workers=0) | throughout |


### Cell 1 — Imports & Seeding

In [14]:
# ==============================================================================
# CELL 1: IMPORTS & SEEDING
# ==============================================================================
import os, random, warnings, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")
NB_VERSION = "2.2-ram"; PIPELINE_STAGE = "NB02"; SEED = 42

def enforce_determinism(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

enforce_determinism(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ {PIPELINE_STAGE} v{NB_VERSION} | Device: {device} | Seed: {SEED}")


✅ NB02 v2.2-ram | Device: cpu | Seed: 42


### Cell 2 — Config & Paths

In [15]:
# ==============================================================================
# CELL 2: CONFIG & PATHS
# ==============================================================================
CONFIG_PATH = Path("config.yaml")
with open(CONFIG_PATH, encoding="utf-8") as f: cfg = yaml.safe_load(f)
print(f"🚀 {cfg['project']['name']} v{cfg['project']['version']}")

BASE_DIR       = Path.cwd()
out_cfg        = cfg.get("output", {})
_ckpt_root     = out_cfg.get("checkpoint_dir", "checkpoints")
_emb_root      = out_cfg.get("embeddings_dir",  "embeddings")
_results_root  = out_cfg.get("results_dir",     "results")
_figures_sub   = out_cfg.get("figures_subdir",  "figures")
CHECKPOINT_DIR  = BASE_DIR / _ckpt_root    / PIPELINE_STAGE
FIGURES_DIR     = BASE_DIR / _results_root / PIPELINE_STAGE / _figures_sub
EMBEDDINGS_DIR  = BASE_DIR / _emb_root     / PIPELINE_STAGE
RAW_DIR         = BASE_DIR / cfg.get("data", {}).get("raw_dir", "data/raw")
for _d in [CHECKPOINT_DIR, FIGURES_DIR, EMBEDDINGS_DIR]: _d.mkdir(parents=True, exist_ok=True)
print(f"   Checkpoints : {_ckpt_root}/{PIPELINE_STAGE}/")
print(f"   Figures     : {_results_root}/{PIPELINE_STAGE}/{_figures_sub}/")
print(f"   Embeddings  : {_emb_root}/{PIPELINE_STAGE}/")

pp=cfg.get("preprocessing",{}); m=cfg.get("model",{}); t=cfg.get("training",{}); s4=cfg.get("stage4_gcn",{})
TARGET_FEATURES = pp.get("variance_filter_top_n", 3000)
BATCH_SIZE      = t.get("batch_size",   64)
LR              = t.get("lr",           5e-4)
WEIGHT_DECAY    = t.get("weight_decay", 0.01)
EPOCHS          = t.get("epochs",       40)
PATIENCE        = t.get("patience",     8)
LATENT_DIM      = s4.get("latent_dim",  m.get("latent_dim",    64))
DROPOUT_RATE    = s4.get("dropout_rate",m.get("dropout_rate",  0.40))
NUM_INTERVALS   = s4.get("num_intervals",m.get("num_intervals", 10))
ENCODER_HIDDEN  = m.get("encoder_hidden", [512, 256])
CROSS_FOLDS     = m.get("cross_folds",  5)
NOISE_FACTOR    = t.get("noise_factor", 0.25)
LOSS_ALPHA      = t.get("structural_loss_weight", 0.5) / (1 + t.get("structural_loss_weight", 0.5))

COHORT_CONFIGS = []
for entry in cfg.get("cohorts", []):
    COHORT_CONFIGS.append({
        "name":  entry["name"],
        "short": entry["short"],          # lowercase from config: lgg/kirc/luad
        "label": entry.get("label", entry["name"]),
        "expr":  RAW_DIR / entry["expression_file"],
        "surv":  RAW_DIR / entry["survival_file"],
    })
if not COHORT_CONFIGS:
    raise ValueError("No cohorts in config.yaml — add a `cohorts:` block.")
# Load NB00 baselines
NB00_CI = {}
for cc in COHORT_CONFIGS:
    sh = cc["short"]
    p  = BASE_DIR/"checkpoints"/"NB00"/f"NB00_{sh}_v3_3_ram_mlp_baseline.pt"
    if p.exists():
        ck = torch.load(p, weights_only=False)
        NB00_CI[sh] = ck.get("honest_ci",{}).get("c_index")
        print(f"   NB00 [{sh}]: {NB00_CI[sh]:.4f}")
    else:
        NB00_CI[sh] = None
        print(f"   NB00 [{sh}]: checkpoint not found — run NB00 first")
print(f"   latent={LATENT_DIM} | intervals={NUM_INTERVALS} | alpha={LOSS_ALPHA:.3f}")


🚀 universal-survival-engine v3.3
   Checkpoints : checkpoints/NB02/
   Figures     : results/NB02/figures/
   Embeddings  : embeddings/NB02/
   NB00 [lgg]: checkpoint not found — run NB00 first
   NB00 [kirc]: checkpoint not found — run NB00 first
   NB00 [luad]: checkpoint not found — run NB00 first
   latent=64 | intervals=10 | alpha=0.333


### Cell 3 — RAM-Optimised Data Loader

In [16]:
# ==============================================================================
# CELL 3: RAM-OPTIMISED DATA LOADER
# ==============================================================================
# RAM management (from KEEP_NB04 template)
# • float32 everywhere — halves expression matrix vs float64
# • del + gc.collect() after every large object
# • torch.cuda.empty_cache() per fold
# • Sequential cohort processing — only one cohort in RAM at a time
# • workers=0 in DataLoader — no subprocess fork RAM copies
# • State dict stored .cpu().clone() — no GPU residual
def _trim(s):
    s = str(s).strip().replace(".", "-").upper()
    return "-".join(s.split("-")[:3]) if s.startswith("TCGA") else s

def _strip_v(ensg): return str(ensg).split(".")[0]

def load_cohort(name, expr_path, surv_path, top_n=3000, label=""):
    """RAM-optimised TCGA loader (float32, nlargest variance, early drop)."""
    print(f"  📂 {name}")

    # ── Survival — read only needed columns ──────────────────────────────────
    clin = pd.read_csv(surv_path, sep="\t", index_col=0)
    clin.index = pd.Index([_trim(i) for i in clin.index])
    clin = clin[~clin.index.duplicated(keep="first")]
    time_col  = next((c for c in ["OS.time","survival_time","time"] if c in clin.columns), None)
    event_col = next((c for c in ["OS","event","status"] if c in clin.columns), None)
    if not time_col or not event_col:
        raise KeyError(f"[{name}] No time/event columns in {list(clin.columns)}")

    # ── Expression — float32 on load, variance filter immediately ────────────
    df = pd.read_csv(expr_path, sep="\t", index_col=0)
    if df.shape[0] > df.shape[1]: df = df.T
    df.columns = pd.Index([_strip_v(c) for c in df.columns])
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    df.index = pd.Index([_trim(i) for i in df.index])
    df = df[~df.index.duplicated(keep="first")].astype(np.float32)  # float32 immediately

    # Variance filter via nlargest — never sorts full array
    top_genes = df.var(axis=0, ddof=0).nlargest(min(top_n, df.shape[1])).index.tolist()
    df = df[top_genes]  # shrink width before any further ops

    # ── Align, filter, clean ─────────────────────────────────────────────────
    common = df.index.intersection(clin.index)
    df     = df.loc[common]
    surv   = clin.loc[common, [time_col, event_col]].copy()
    surv.columns = ["t", "e"]
    valid  = surv["t"].notna() & surv["e"].notna() & (surv["t"] > 0)
    df     = df.loc[valid].dropna(axis=1).astype(np.float32)
    surv   = surv.loc[valid].astype(np.float32)

    # Cast survival to float32
    y_time  = surv["t"].values.astype(np.float32)
    y_event = surv["e"].values.astype(np.float32)

    ram_mb = df.memory_usage(deep=True).sum() / 1e6
    n_drop = int((~valid).sum())
    print(f"     {len(df)} patients | {df.shape[1]:,} genes | "
          f"{int(y_event.sum())} events ({100*y_event.mean():.1f}%) | "
          f"RAM: {ram_mb:.1f} MB" + (f" | dropped {n_drop} neg-time" if n_drop else ""))

    return {"name": name, "label": label or name,
            "X_raw": df.values.astype(np.float32),
            "gene_cols": df.columns.tolist(),
            "y_time": y_time, "y_event": y_event}

print("✅ load_cohort() defined.")


✅ load_cohort() defined.


### Cell 4 — DAE Model & Loss

In [17]:
# ==============================================================================
# CELL 4: DAE MODEL & LOSS
# ==============================================================================
class SurvivalDataset(Dataset):
    def __init__(self, X, times, events):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.times=torch.tensor(times,dtype=torch.float32)
        self.events=torch.tensor(events,dtype=torch.float32)
    def __len__(self): return len(self.times)
    def __getitem__(self, idx): return self.X[idx], self.times[idx], self.events[idx]

class DiscreteHazardDAE(nn.Module):
    def __init__(self, input_dim, latent_dim=64, hidden=None, num_intervals=10, dropout=0.40):
        super().__init__()
        hidden = hidden or [512, 256]
        enc, in_d = [], input_dim
        for h in hidden:
            enc += [nn.Linear(in_d,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]; in_d = h
        enc += [nn.Linear(in_d,latent_dim), nn.BatchNorm1d(latent_dim), nn.ReLU()]
        self.encoder = nn.Sequential(*enc)
        dec, in_d = [], latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(in_d,h), nn.ReLU()]; in_d = h
        dec += [nn.Linear(in_d, input_dim)]
        self.decoder = nn.Sequential(*dec)
        self.hazard_head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(latent_dim//2, num_intervals), nn.Sigmoid()
        )
    def forward(self, x, noise_factor=0.0, training=False):
        if training and noise_factor > 0: x = x + torch.randn_like(x) * noise_factor
        z = self.encoder(x); return z, self.decoder(z), self.hazard_head(z)

class DiscreteLifespanLoss(nn.Module):
    """
    Vectorised discrete-time survival loss (v2.2).

    Replaces the O(N·K) Python loop from v2.1 with three tensor operations:

      1. bin_idx  — digitise all patient times at once            [N]
      2. mask_pre — lower-triangular bool: bin b < patient bin    [N, K]
      3. mask_at  — one-hot: bin b == patient bin                 [N, K]

    The NLL is then a fully-batched dot product — no Python iteration
    over patients or intervals. The implementation is fp16-safe and
    therefore compatible with torch.autocast (AMP).

    Loss = alpha * recon_MSE + (1 - alpha) * nll_survival
    """
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.mse   = nn.MSELoss()

    def forward(self, x_hat, x, hazards, times, events, time_bins):
        # ── Reconstruction term ───────────────────────────────────────────────
        recon = self.mse(x_hat, x)

        # ── Bin assignment (vectorised digitise) ──────────────────────────────
        # time_bins is a numpy array of K interval edges; convert once
        bins_t = torch.as_tensor(time_bins, dtype=hazards.dtype, device=hazards.device)
        # digitize: count how many bin edges each time exceeds → interval index
        # shape [N, K] broadcast: times [N,1] >= bins_t [1,K]
        bin_idx = (times.unsqueeze(1) >= bins_t.unsqueeze(0)).sum(dim=1)   # [N]
        bin_idx = bin_idx.clamp(0, hazards.size(1) - 1)

        K = hazards.size(1)
        k_range = torch.arange(K, device=hazards.device)                   # [K]

        # ── Survival contribution (bins before the event bin) ─────────────────
        # mask_pre[i, b] = True when b < bin_idx[i]
        mask_pre = k_range.unsqueeze(0) < bin_idx.unsqueeze(1)             # [N, K]
        # log(1 - h) for all pre-event bins, summed per patient
        log_surv = (torch.log(1.0 - hazards.clamp(1e-7, 1 - 1e-7)) * mask_pre).sum(dim=1)  # [N]

        # ── Event/censoring contribution (at the event bin) ───────────────────
        mask_at  = k_range.unsqueeze(0) == bin_idx.unsqueeze(1)            # [N, K]
        h_at     = (hazards * mask_at).sum(dim=1).clamp(1e-7, 1 - 1e-7)   # [N]
        # events==1: log(h_at);  events==0: log(1 - h_at)
        log_at   = torch.where(events == 1, torch.log(h_at), torch.log(1.0 - h_at))  # [N]

        nll = -(log_surv + log_at).mean()

        total = self.alpha * recon + (1 - self.alpha) * nll
        return total, recon, nll

print(f"✅ DiscreteHazardDAE & vectorised DiscreteLifespanLoss defined.")
print(f"   latent={LATENT_DIM} | intervals={NUM_INTERVALS} | alpha={LOSS_ALPHA:.3f}")
print(f"   Loss: O(N·K) Python loop → 3 tensor ops (fp16-safe)")


✅ DiscreteHazardDAE & vectorised DiscreteLifespanLoss defined.
   latent=64 | intervals=10 | alpha=0.333
   Loss: O(N·K) Python loop → 3 tensor ops (fp16-safe)


### Cell 5 — RAM-Optimised Fold Training

In [18]:
# ==============================================================================
# CELL 5: RAM-OPTIMISED FOLD TRAINING
# ==============================================================================
def train_one_fold(X_tr_raw, X_val_raw, t_tr, t_val, e_tr, e_val, label=""):
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr_raw)
    X_val  = scaler.transform(X_val_raw)
    # free raw slices — X_tr/X_val are the only copies we need
    del X_tr_raw, X_val_raw; gc.collect()

    ev_t = t_tr[e_tr == 1]
    time_bins = (np.quantile(ev_t, np.linspace(0.1,1.0,NUM_INTERVALS)) if len(ev_t)>=NUM_INTERVALS
                 else np.quantile(t_tr, np.linspace(0.1,1.0,NUM_INTERVALS)))

    train_ldr = DataLoader(SurvivalDataset(X_tr, t_tr, e_tr),
                           batch_size=BATCH_SIZE, shuffle=True,
                           drop_last=True, num_workers=0)   # workers=0 — no fork RAM

    model = DiscreteHazardDAE(X_tr.shape[1], LATENT_DIM, ENCODER_HIDDEN,
                               NUM_INTERVALS, DROPOUT_RATE).to(device)
    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lf    = DiscreteLifespanLoss(alpha=LOSS_ALPHA)
    best_c=-1.0; best_st=None; best_bins=time_bins.copy(); wait=0; ep_l=[]

    for epoch in range(1, EPOCHS+1):
        model.train(); el=0.; en=0
        for feats,times,evts in train_ldr:
            feats,times,evts = feats.to(device),times.to(device),evts.to(device)
            opt.zero_grad()
            _,x_hat,hazards = model(feats, noise_factor=NOISE_FACTOR, training=True)
            loss,_,_ = lf(x_hat, feats, hazards, times, evts, time_bins)
            loss.backward(); opt.step(); el+=loss.item(); en+=1
        ep_l.append(el/max(en,1))
        model.eval()
        with torch.no_grad():
            _,_,h = model(torch.tensor(X_val,dtype=torch.float32).to(device))
            # Expected bin index (negated) — matches NB04 risk convention.
            # Patients with hazard mass shifted toward early bins → higher
            # expected bin → lower negated value → correctly ranked as high-risk.
            bins_w = torch.arange(h.size(1), dtype=h.dtype, device=h.device)
            risk   = -(h * bins_w).sum(dim=1).cpu().numpy()
        try:    c = concordance_index(t_val, risk, e_val)
        except: c = 0.5
        if c > best_c:
            best_c=c; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}
            best_bins=time_bins.copy(); wait=0
        else:
            wait+=1
            if wait>=PATIENCE: print(f"  {label}: early stop ep {epoch}"); break

    # Free model + optimiser immediately
    del model, opt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"val_c":best_c,"state":best_st,"scaler":scaler,"time_bins":best_bins,"ep_losses":ep_l}

print("train_one_fold() ready.")


train_one_fold() ready.


### Cell 6 — Sequential Cohort CV Loop

In [19]:
# ==============================================================================
# CELL 6: SEQUENTIAL COHORT LOOP — one cohort in RAM at a time
# ==============================================================================
ALL_CV = []   # lightweight summaries only

for cc in COHORT_CONFIGS:
    name = cc["name"]; sh = cc["short"]
    print(f"\n{'='*65}\n  {name}\n{'='*65}")

    # Load — float32, nlargest, dropna
    cohort = load_cohort(name, cc["expr"], cc["surv"], top_n=TARGET_FEATURES, label=cc["label"])
    X_raw  = cohort["X_raw"]; y_time = cohort["y_time"]; y_event = cohort["y_event"]

    kf = KFold(n_splits=CROSS_FOLDS, shuffle=True, random_state=SEED)
    fold_c=[]; fold_losses=[]; best_val_c=-1.0; best_out=None

    for fold,(tr_idx,val_idx) in enumerate(kf.split(X_raw)):
        label = f"Fold {fold+1}"
        out   = train_one_fold(
            X_raw[tr_idx].copy(), X_raw[val_idx].copy(),   # .copy() avoids view RAM retention
            y_time[tr_idx], y_time[val_idx],
            y_event[tr_idx], y_event[val_idx], label)
        fold_c.append(out["val_c"]); fold_losses.append(out["ep_losses"])
        if out["val_c"] > best_val_c: best_val_c=out["val_c"]; best_out=out
        print(f"  Fold {fold+1}: C={out['val_c']:.4f}")
        gc.collect()

    mean_c=float(np.mean(fold_c)); std_c=float(np.std(fold_c))
    print(f"  Mean: {mean_c:.4f} ± {std_c:.4f}")

    # Checkpoint
    ckpt_name = f"NB02_{sh}_dae_discrete.pt"
    torch.save({
        "pipeline_stage":PIPELINE_STAGE,"nb_version":NB_VERSION,"cohort":sh,
        "model_state_dict":best_out["state"],
        "config":{"input_dim":X_raw.shape[1],"latent_dim":LATENT_DIM,
                  "hidden":ENCODER_HIDDEN,"num_intervals":NUM_INTERVALS,"dropout":DROPOUT_RATE},
        "time_bins":best_out["time_bins"],"fold_c_indices":fold_c,
        "mean_c":mean_c,"std_c":std_c,"nb00_baseline_ci":NB00_CI.get(sh),
    }, CHECKPOINT_DIR / ckpt_name)
    print(f"  💾 checkpoints/NB02/{ckpt_name}")

    # ── Embedding export ──────────────────────────────────────────────────
    # Rebuild model from best fold weights
    _m = DiscreteHazardDAE(X_raw.shape[1], LATENT_DIM, ENCODER_HIDDEN,
                           NUM_INTERVALS, DROPOUT_RATE).to(device)
    _m.load_state_dict(best_out["state"]); _m.eval()

    # — A. Held-out embeddings (leakage audit) —
    # Scaler from best fold's train split — patients in val/test were
    # never seen during fitting. Use these for any downstream survival metric.
    X_heldout_sc = best_out["scaler"].transform(X_raw).astype(np.float32)
    with torch.no_grad():
        z_heldout, _, h_heldout = _m(torch.tensor(X_heldout_sc).to(device))
        z_heldout = z_heldout.cpu().numpy()
        bins_w    = torch.arange(h_heldout.size(1), dtype=h_heldout.dtype, device=h_heldout.device)
        risk_heldout = -(h_heldout * bins_w).sum(dim=1).cpu().numpy()
    del X_heldout_sc, h_heldout

    z_cols = [f"z_{i}" for i in range(LATENT_DIM)]
    df_heldout = pd.DataFrame(z_heldout, columns=z_cols)
    df_heldout["risk_score"]    = risk_heldout
    df_heldout["survival_time"] = y_time
    df_heldout["event"]         = y_event
    # scaler_note: fitted on best fold train split only — held-out patients unseen
    heldout_path = EMBEDDINGS_DIR / f"NB02_{sh}_heldout_latents.csv"
    df_heldout.to_csv(heldout_path, index=False)
    print(f"  📁 embeddings/NB02/NB02_{sh}_heldout_latents.csv  "
          f"(leakage-safe — fold train scaler)")
    del z_heldout, risk_heldout, df_heldout

    # — B. Full-cohort embeddings (representation / visualisation only) —
    # Scaler refit on ALL patients — not for any survival metric or audit.
    # Use only for UMAP, clustering, or cross-cohort representation comparison.
    sc_full = StandardScaler()
    X_full_sc = sc_full.fit_transform(X_raw).astype(np.float32)
    with torch.no_grad():
        z_full, _, h_full = _m(torch.tensor(X_full_sc).to(device))
        z_full = z_full.cpu().numpy()
        risk_full = -(h_full * bins_w).sum(dim=1).cpu().numpy()
    del X_full_sc, h_full

    df_full = pd.DataFrame(z_full, columns=z_cols)
    df_full["risk_score"]    = risk_full
    df_full["survival_time"] = y_time
    df_full["event"]         = y_event
    # scaler_note: refit on full cohort — representation only, NOT for metrics
    full_path = EMBEDDINGS_DIR / f"NB02_{sh}_fullcohort_latents.csv"
    df_full.to_csv(full_path, index=False)
    print(f"  📁 embeddings/NB02/NB02_{sh}_fullcohort_latents.csv  "
          f"(full-cohort scaler — representation only, not for metrics)")
    del _m, z_full, risk_full, df_full, sc_full
    gc.collect()

    # KM figure — use best_out model temporarily then delete
    _m = DiscreteHazardDAE(X_raw.shape[1],LATENT_DIM,ENCODER_HIDDEN,NUM_INTERVALS,DROPOUT_RATE).to(device)
    _m.load_state_dict(best_out["state"]); _m.eval()
    X_sc = best_out["scaler"].transform(X_raw)
    with torch.no_grad():
        _,_,h = _m(torch.tensor(X_sc,dtype=torch.float32).to(device))
        bins_w = torch.arange(h.size(1), dtype=h.dtype, device=h.device)
        risk   = -(h * bins_w).sum(dim=1).cpu().numpy()
    del _m, X_sc; gc.collect()
    med=np.median(risk); hi=risk>=med; lo=~hi
    fig,ax=plt.subplots(figsize=(7,5))
    KaplanMeierFitter(label=f"High(n={hi.sum()})").fit(y_time[hi],y_event[hi]).plot_survival_function(ax=ax,color="#d62728",ci_show=True)
    KaplanMeierFitter(label=f"Low(n={lo.sum()})").fit(y_time[lo],y_event[lo]).plot_survival_function(ax=ax,color="#1f77b4",ci_show=True)
    if hi.sum()>=5 and lo.sum()>=5:
        from lifelines.statistics import logrank_test as lrt
        p=lrt(y_time[hi],y_time[lo],y_event[hi],y_event[lo]).p_value
        ax.text(0.97,0.05,f"p={p:.4f}",transform=ax.transAxes,ha="right",fontsize=9,
                bbox=dict(boxstyle="round",fc="white",alpha=0.8))
    ax.set(title=f"NB02 [{name}]  C={mean_c:.3f}±{std_c:.3f}",xlabel="Days",ylabel="P(survive)")
    ax.grid(True,ls="--",alpha=0.3)
    fig.tight_layout(); fig.savefig(FIGURES_DIR/f"NB02_{sh}_km.png",dpi=150,bbox_inches="tight")
    plt.close(fig)   # close immediately
    print(f"  ✓ results/NB02/figures/NB02_{sh}_km.png")

    ALL_CV.append({"name":name,"short":sh,"label":cc["label"],
                   "mean_c":mean_c,"std_c":std_c,"fold_c":fold_c,
                   "nb00_ci":NB00_CI.get(sh)})

    # Free ALL cohort RAM before next
    del cohort, X_raw, y_time, y_event, best_out, fold_c, fold_losses, risk
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  🧹 [{name}] RAM freed.")

# Summary
print(f"\n{'='*60}\nNB02 SUMMARY")
print(f"  {'Cohort':<12} {'Mean C':>8} {'Std':>7}  {'NB00':>8}  Δ")
print("  "+"-"*48)
for r in ALL_CV:
    nb0=r["nb00_ci"]; delta=f"{r['mean_c']-nb0:+.4f}" if nb0 else "n/a"
    flag="✅" if nb0 and r["mean_c"]>nb0 else ("❌" if nb0 else "—")
    print(f"  {r['name']:<12} {r['mean_c']:>8.4f} {r['std_c']:>7.4f}  "
          f"{nb0 if nb0 else 'n/a':>8}  {delta} {flag}")
print(f"{'='*60}")



  TCGA-LGG
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time
  Fold 1: early stop ep 36
  Fold 1: C=0.8535
  Fold 2: early stop ep 23
  Fold 2: C=0.8497
  Fold 3: early stop ep 9
  Fold 3: C=0.8184
  Fold 4: early stop ep 10
  Fold 4: C=0.8210
  Fold 5: early stop ep 10
  Fold 5: C=0.6763
  Mean: 0.8038 ± 0.0654
  💾 checkpoints/NB02/NB02_lgg_dae_discrete.pt
  📁 embeddings/NB02/NB02_lgg_heldout_latents.csv  (leakage-safe — fold train scaler)
  📁 embeddings/NB02/NB02_lgg_fullcohort_latents.csv  (full-cohort scaler — representation only, not for metrics)
  ✓ results/NB02/figures/NB02_lgg_km.png
  🧹 [TCGA-LGG] RAM freed.

  TCGA-KIRC
  📂 TCGA-KIRC
     531 patients | 3,000 genes | 175 events (33.0%) | RAM: 6.4 MB
  Fold 1: early stop ep 18
  Fold 1: C=0.7548
  Fold 2: early stop ep 24
  Fold 2: C=0.7426
  Fold 3: early stop ep 9
  Fold 3: C=0.6430
  Fold 4: early stop ep 16
  Fold 4: C=0.6452
  Fold 5: early stop ep 10
  Fold 5: C=0.6381


In [20]:
# ==============================================================================
# CELL 6b: DEAD TOLL + SURVIVAL COUNT + KM MILESTONE TABLE  (added v2.2-ram+)
# Must run after the main cohort loop (ALL_CV populated, checkpoints saved).
# ==============================================================================
from lifelines import KaplanMeierFitter
import pandas as pd

MILESTONES_DAYS = [365, 3*365, 5*365]

print(f"\n{'='*72}")
print("  DEAD TOLL & SURVIVAL TABLE — NB02 DAE Baseline")
print(f"{'='*72}")

for cc, cv in zip(COHORT_CONFIGS, ALL_CV):
    name   = cc["name"]; sh = cc["short"]
    ckpt_p = CHECKPOINT_DIR / f"NB02_{sh}_dae_discrete.pt"
    if not ckpt_p.exists():
        print(f"  [{name}] checkpoint not found — skipping"); continue

    # ── A. Raw event array ────────────────────────────────────────────────────
    cohort  = load_cohort(name, cc["expr"], cc["surv"],
                          top_n=TARGET_FEATURES, label=cc["label"])
    y_time  = cohort["y_time"]
    y_event = cohort["y_event"]

    dead_toll      = int(y_event.sum())
    survival_count = int((1 - y_event).sum())
    n_total        = len(y_event)
    print(f"\n  [{name}]  N={n_total}")
    print(f"    Dead Toll      (Σ δᵢ = 1) : {dead_toll}  ({100*dead_toll/n_total:.1f}%)")
    print(f"    Survival Count (Σ 1-δᵢ)   : {survival_count}  ({100*survival_count/n_total:.1f}%)")

    # ── B. Risk scores from checkpoint ────────────────────────────────────────
    ck  = torch.load(ckpt_p, weights_only=False)
    cfg = ck["config"]
    _m  = DiscreteHazardDAE(cfg["input_dim"], cfg["latent_dim"], cfg["hidden"],
                             cfg["num_intervals"], cfg["dropout"]).to(device)
    _m.load_state_dict(ck["model_state_dict"]); _m.eval()
    X_raw = cohort["X_raw"]
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler(); sc.fit(X_raw)   # refit on full cohort (no held-out leakage for display)
    X_sc = sc.transform(X_raw)
    with torch.no_grad():
        _, _, h = _m(torch.tensor(X_sc, dtype=torch.float32).to(device))
        bins_w  = torch.arange(h.size(1), dtype=h.dtype, device=h.device)
        risk    = -(h * bins_w).sum(dim=1).cpu().numpy()
    del _m, X_sc; gc.collect()

    # ── C. Stratify ───────────────────────────────────────────────────────────
    thresh = np.median(risk)
    hi_m   = risk >= thresh
    lo_m   = ~hi_m

    # ── D. KM milestone table ─────────────────────────────────────────────────
    rows = []
    for group_label, mask_g in [("High-Risk", hi_m), ("Low-Risk", lo_m), ("Overall", np.ones(len(risk), dtype=bool))]:
        kmf = KaplanMeierFitter(label=group_label)
        kmf.fit(y_time[mask_g], y_event[mask_g])
        et  = kmf.event_table
        for ms in MILESTONES_DAYS:
            yr  = ms // 365
            sub = et[et.index <= ms]
            at_risk    = int(sub["at_risk"].iloc[-1])  if len(sub) else int(mask_g.sum())
            n_events   = int(sub["observed"].sum())
            n_censored = int(sub["censored"].sum())
            sf_val     = float(kmf.survival_function_at_times([ms]).iloc[0])
            rows.append({"Cohort": name, "Group": group_label,
                         "Year": f"Y{yr}",
                         "At-Risk": at_risk,
                         "Events (Dead)": n_events,
                         "Censored": n_censored,
                         "S(t)": round(sf_val, 4)})

    tbl = pd.DataFrame(rows)
    print()
    print(tbl.to_string(index=False))

    del cohort, X_raw, risk; gc.collect()

print(f"\n{'='*72}")



  DEAD TOLL & SURVIVAL TABLE — NB02 DAE Baseline
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time

  [TCGA-LGG]  N=512
    Dead Toll      (Σ δᵢ = 1) : 126  (24.6%)
    Survival Count (Σ 1-δᵢ)   : 386  (75.4%)

  Cohort     Group Year  At-Risk  Events (Dead)  Censored   S(t)
TCGA-LGG High-Risk   Y1      226              0        31 1.0000
TCGA-LGG High-Risk   Y3      111              0       146 1.0000
TCGA-LGG High-Risk   Y5       46              1       210 0.9904
TCGA-LGG  Low-Risk   Y1      172             26        60 0.8781
TCGA-LGG  Low-Risk   Y3       49             75       133 0.4898
TCGA-LGG  Low-Risk   Y5       22             97       138 0.2507
TCGA-LGG   Overall   Y1      397             26        91 0.9419
TCGA-LGG   Overall   Y3      159             75       279 0.7771
TCGA-LGG   Overall   Y5       67             98       348 0.6245
  📂 TCGA-KIRC
     531 patients | 3,000 genes | 175 events (33.0%) | RAM: 6.4 MB

  [TC